# 15 — Out-of-Distribution Evaluation (EDF-S)

All prior JAISP metrics are on **ECDFS tract 5063** (in-distribution). This notebook evaluates the frozen v10 stack on **EDF-S (Euclid Deep Field South, tract 2394)** with **no retraining**, to test the foundation's actual value proposition: transfer to a new field.

Data: `data/edf_s_ood/` — 72 paired Euclid+Rubin tiles. Note the EDF-S Euclid tiles carry **no `var_*` maps**; both the detection loader and the astrometry dataset fall back to a MAD-based RMS estimate, so the two stages share one noise model.

Pipeline (each step mirrors the ECDFS production config exactly):
1. **Detection** — CenterNet v10 export → `data/detection_labels/centernet_v10_edfs_thresh03.pt`
2. **Astrometry** — joint canonical anchors via the latent-position head
3. **Comparison** — cross-instrument MAE/median vs the ECDFS reference (25.4 MAE / 11.3 median)

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'io' else Path.cwd()
OOD_CACHE   = ROOT / 'data/detection_labels/centernet_v10_edfs_thresh03.pt'
ECDFS_CACHE = ROOT / 'data/detection_labels/centernet_v10_790_thresh03.pt'
EUCLID_DIR  = ROOT / 'data/edf_s_ood/euclid_tiles_edfs'
print('root:', ROOT)
print('ood cache exists:', OOD_CACHE.exists())

## Step 1 — Detection on EDF-S

Export command (run once from the repo root; mirrors the production cache config — fused CenterNet, conf 0.3, NMS 7, spike-veto off, no bright-rescue):

```bash
python models/detection/run_centernet_detections.py \
    --encoder_ckpt   models/checkpoints/jaisp_v10_warmstart/checkpoint_best.pt \
    --centernet_ckpt checkpoints/centernet_v10_uncertain_synth_r2/centernet_best.pt \
    --rubin_dir      data/edf_s_ood/rubin_tiles_edfs \
    --euclid_dir     data/edf_s_ood/euclid_tiles_edfs \
    --out            data/detection_labels/centernet_v10_edfs_thresh03.pt \
    --conf_threshold 0.3 --nms_kernel 7 --spike_veto_radius 0 --spike_veto_width 0
```

In [ ]:
ood = torch.load(OOD_CACHE, map_location='cpu', weights_only=False)
labels = ood['labels']
ood_n = np.array([v[0].shape[0] for v in labels.values()])
print('config:', ood['config'].get('n_errors'), 'errors,', len(labels), 'tiles')
print('det/tile: mean %.1f  median %d  min %d  max %d' % (
    ood_n.mean(), np.median(ood_n), ood_n.min(), ood_n.max()))

### 1a. Overlay detections on VIS
Red circles should sit on real sources, not blank sky / noise spikes. Bright or extended objects should not be systematically missed.

In [ ]:
def zscale(img, lo=1.0, hi=99.5):
    f = img[np.isfinite(img)]
    return np.percentile(f, [lo, hi])

SHOW = ['tile_x00000_y01536', 'tile_x00256_y02048', 'tile_x00512_y02304']
show = [s for s in SHOW if s in labels] or list(labels.keys())[:3]

fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 6))
axes = np.atleast_1d(axes)
for ax, stem in zip(axes, show):
    ed = np.load(EUCLID_DIR / f'{stem}_euclid.npz', allow_pickle=True)
    vis = np.nan_to_num(np.asarray(ed['img_VIS'], dtype=np.float32))
    H, W = vis.shape
    vlo, vhi = zscale(vis)
    ax.imshow(vis, origin='lower', cmap='gray', vmin=vlo, vmax=vhi)
    xy, _ = labels[stem]
    ax.scatter(xy[:, 0] * W, xy[:, 1] * H, s=14, facecolors='none',
               edgecolors='red', linewidths=0.6)
    ax.set_title(f'{stem}\n{xy.shape[0]} detections')
    ax.set_xlim(0, W); ax.set_ylim(0, H)
plt.tight_layout(); plt.show()

### 1b. Detection density vs ECDFS
A uniform shift in counts/tile is expected for a deeper/denser field; a long tail driven by a few tiles flags crowding artifacts.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.linspace(0, max(1100, ood_n.max()), 40)
ax.hist(ood_n, bins=bins, density=True, alpha=0.6, color='C1',
        label=f'EDF-S OOD (n={len(ood_n)})')
if ECDFS_CACHE.exists():
    ec = torch.load(ECDFS_CACHE, map_location='cpu', weights_only=False)
    ec_n = np.array([v[0].shape[0] for v in ec['labels'].values()])
    ax.hist(ec_n, bins=bins, density=True, alpha=0.5, color='C0',
            label=f'ECDFS prod (n={len(ec_n)})')
    print('ECDFS det/tile: mean %.1f median %d' % (ec_n.mean(), np.median(ec_n)))
ax.axvline(ood_n.mean(), color='C1', ls='--')
ax.set_xlabel('detections / tile'); ax.set_ylabel('density')
ax.set_title('Detection density: EDF-S OOD vs ECDFS'); ax.legend()
plt.tight_layout(); plt.show()

## Step 2 — Astrometry on EDF-S (joint canonical anchors)
_To be filled in after the detection overlay is confirmed clean._